# Polymorphism in Python

Goal: understand why polymorphism exists, duck typing, method overriding as polymorphism, and where it already shows up in Python's built-ins.

## 1. What is Polymorphism?

**Why we need it:** say you have a `Circle`, a `Square`, and a `Triangle`, each with its own `area()` formula. Without polymorphism, code that needs to total up areas would have to know each shape's exact type and call a different calculation for each one — a long chain of `if isinstance(shape, Circle): ... elif isinstance(shape, Square): ...`. Every time a new shape type is added, that chain has to be found and extended everywhere it appears.

**Use case:** anywhere you want to write code once that works on *any* object exposing the right method or behavior, regardless of its specific type — totaling areas of mixed shapes, calling `.pay()` on a list of different employee types, calling `.render()` on a list of different UI widgets.

**How to implement:** give every type in the group a method with the **same name** (like `area()`), each implemented to do the right thing for that type. Calling code then just calls `shape.area()` without caring or checking which shape it actually is — Python figures out, at the moment the call happens, which specific version of `area()` to run based on the object's actual type. "Poly" + "morph" literally means "many forms" — one call, many possible behaviors depending on the object.

In [1]:
import math


class Circle:
    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return math.pi * self.radius ** 2


class Square:
    def __init__(self, side):
        self.side = side

    def area(self):
        return self.side ** 2


class Triangle:
    def __init__(self, base, height):
        self.base = base
        self.height = height

    def area(self):
        return 0.5 * self.base * self.height


shapes = [Circle(3), Square(4), Triangle(6, 5)]

# same call, shape.area(), works for every type -- no isinstance chain needed
for shape in shapes:
    print(f"{type(shape).__name__}: {shape.area():.2f}")

total = sum(shape.area() for shape in shapes)
print(f"total area: {total:.2f}")

Circle: 28.27
Square: 16.00
Triangle: 15.00
total area: 59.27


## 2. Duck Typing

**Why we need it:** notice that `Circle`, `Square`, and `Triangle` above don't share a common parent class, and nothing declared "these three are related." The loop calling `shape.area()` still worked anyway. In many languages, polymorphism requires the types to formally share an interface or base class. Python doesn't require that at all — it just tries the call, and if the object has that method, it works.

**Use case:** writing a function that accepts "anything with a `.read()` method" or "anything with a `.area()` method" without forcing unrelated classes into an artificial shared inheritance hierarchy just to satisfy a type check.

**How to implement:** just call the method you need on the object, without checking its type first. This is called **duck typing**, from "if it walks like a duck and quacks like a duck, it's a duck" — Python doesn't check what the object officially *is*, only whether it can actually *do* what you're asking. If the object doesn't have the method, you find out at the moment you call it, as a normal `AttributeError` — not before.

In [2]:
class Duck:
    def make_sound(self):
        return "Quack!"


class Person:
    def make_sound(self):
        return "I'm quacking like a duck!"


class Car:
    def honk(self):          # note: no make_sound() at all
        return "Beep!"


def make_it_quack(thing):
    return thing.make_sound()   # no type check -- just calls the method and hopes it exists


for entity in [Duck(), Person()]:
    print(make_it_quack(entity))     # both work: neither shares a base class, both have the method

try:
    make_it_quack(Car())              # Car has no make_sound() -- fails at call time
except AttributeError as e:
    print(f"AttributeError: {e}")

Quack!
I'm quacking like a duck!
AttributeError: 'Car' object has no attribute 'make_sound'


## 3. Polymorphism via Inheritance (Method Overriding)

**Why we need it:** duck typing works well when classes are genuinely unrelated. But often the different types *are* clearly related — different kinds of employees, different kinds of notifications — and you want that relationship documented in the code, plus the ability to share common behavior (as inheritance already lets you do), while still letting each subclass respond differently to the same method call.

**Use case:** a list of different `Employee` subclasses (`Manager`, `Developer`, `Intern`), all sharing the base `Employee` interface, where calling `.describe()` on any of them gives type-appropriate output without the calling code needing to know which subclass it's dealing with.

**How to implement:** this is the same **method overriding** from the inheritance notebook, looked at from a different angle. Define a method in the base class, then override it in each subclass with behavior specific to that subclass. Code that loops over a mix of these objects and calls the shared method name automatically gets the right version for each one — that's polymorphism, achieved through an inheritance hierarchy instead of duck typing's "no shared type needed at all."

In [3]:
class Employee:
    def __init__(self, name):
        self.name = name

    def describe(self):
        return f"{self.name} is an employee"


class Manager(Employee):
    def describe(self):                      # overridden: different behavior for this subclass
        return f"{self.name} manages a team"


class Developer(Employee):
    def describe(self):
        return f"{self.name} writes code"


class Intern(Employee):
    pass   # no override -- falls back to Employee.describe()


staff = [Manager("Alice"), Developer("Bob"), Intern("Carol")]

for person in staff:
    print(person.describe())   # same call, different result depending on the actual subclass

Alice manages a team
Bob writes code
Carol is an employee


## 4. Polymorphism You've Already Been Using

**Why we need it (or rather, why it's worth noticing):** polymorphism isn't a special technique reserved for custom classes — it's already how Python's own built-in operators and functions behave. Recognizing that makes the concept feel far less abstract.

**Use case:** `len(...)` works on a `str`, a `list`, a `dict`, and a `tuple` — four completely different internal structures, one shared call. `+` adds numbers, but also concatenates strings and joins lists — the same operator, different behavior depending on the operands' types.

**How it works:** every one of these built-in types implements the appropriate underlying method (`__len__` for `len()`, `__add__` for `+`) in its own way. `len(x)` is really just Python calling `x.__len__()` for you. This is exactly the same mechanism as `shape.area()` from section 1 — just built into the language itself instead of written by you. (These double-underscore methods are called "magic methods," and get their own dedicated notebook — including how to make `+` work on your own classes, which is "operator overloading.")

In [4]:
for value in ["hello", [1, 2, 3], {"a": 1, "b": 2}, (1, 2, 3, 4)]:
    print(f"{type(value).__name__}: len={len(value)}")   # same len() call, different __len__ each time

print("ab" + "cd")        # __add__ on str -> concatenation
print([1, 2] + [3, 4])    # __add__ on list -> joining
print(3 + 4)               # __add__ on int -> arithmetic

print("hello".__len__())  # len(x) is really just calling this for you
print("hello".__len__() == len("hello"))

str: len=5
list: len=3
dict: len=2
tuple: len=4
abcd
[1, 2, 3, 4]
7
5
True


## 5. Putting It Together: a Notification System

**Why we need it:** to see both flavors of polymorphism from this notebook working side by side — an inheritance hierarchy for the notification types that share structure, plus one unrelated, duck-typed class thrown into the same list — and confirm the same calling code handles all of them without caring which is which.

**Use case:** a notification system that needs to `send()` an `Email` or an `SMS`, plus a third-party `SlackWebhook` object that also knows how to `send()` but was never written to share any base class with the others.

**How it's implemented here:** `Notification` is the shared base class; `Email` and `SMS` override `send()`. `SlackWebhook` is completely unrelated — it just happens to also have a `send()` method. The loop at the bottom calls `notification.send()` on all three without an `isinstance` check anywhere, relying on inheritance-based polymorphism for the first two and duck typing for the third.

In [5]:
class Notification:
    def __init__(self, recipient, message):
        self.recipient = recipient
        self.message = message

    def send(self):
        return f"Generic notification to {self.recipient}: {self.message}"


class Email(Notification):
    def send(self):
        return f"Emailing {self.recipient}: {self.message}"


class SMS(Notification):
    def send(self):
        return f"Texting {self.recipient}: {self.message}"


class SlackWebhook:
    """Unrelated to Notification -- no shared base class at all."""
    def __init__(self, channel, message):
        self.channel = channel
        self.message = message

    def send(self):
        return f"Posting to #{self.channel}: {self.message}"


notifications = [
    Email("alice@example.com", "Your order shipped"),
    SMS("+1-555-0100", "Your OTP is 4821"),
    SlackWebhook("general", "Deploy finished"),
]

for notification in notifications:
    print(notification.send())   # one call, three different implementations behind it

Emailing alice@example.com: Your order shipped
Texting +1-555-0100: Your OTP is 4821
Posting to #general: Deploy finished
